In [1]:
# ============================================================
# Google Colab setup
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "CI_tutorial"
REPO_URL = "https://github.com/tiksato/CI_tutorial.git"
REPO_DIR = Path("/content") / REPO_NAME

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    os.chdir(REPO_DIR)

    # Install dependencies from pyproject.toml
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", "."],
        check=True
    )

    # Then move to notebooks directory if desired
    os.chdir(REPO_DIR / "notebooks")

# Add parent of CI_tutorial to sys.path
# This allows imports like:
# from CI_tutorial.modules... import ...
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    REPO_DIR = NOTEBOOK_DIR.parent
else:
    REPO_DIR = NOTEBOOK_DIR

PARENT_DIR = REPO_DIR.parent

if str(PARENT_DIR) not in sys.path:
    sys.path.insert(0, str(PARENT_DIR))

print("Current directory:", Path.cwd())
print("Repository:", REPO_DIR)
print("Python path added:", PARENT_DIR)
print("Setup completed.")

Current directory: /Users/sato/Downloads/test_CI_tutorial/CI_tutorial/notebooks
Repository: /Users/sato/Downloads/test_CI_tutorial/CI_tutorial
Python path added: /Users/sato/Downloads/test_CI_tutorial
Setup completed.


In [2]:
import time
import math
import numpy as np
from pyscf import gto
from CI_tutorial.modules.misc_utils.pyscf_tools import get_integrals_rhf
from CI_tutorial.modules.misc_utils.pyscf_tools import davidson_restricted_fullci_mask as davidson_pyscf
from CI_tutorial.modules.misc_utils.matrix_utilities import davidson
from CI_tutorial.modules.ormas_tools.ormas2 import ORMAS2

# Maximum CI dimension for memory safety                                                                                         
max_dim_ormas     =  200000000 # 0.2 billion                                                                                     
max_dim_fci_pyscf = 1000000000 # 1 billion

In [3]:
# We consider an Ar atom, with (9,9) electrons in total,                                         
# with 6-31G** basis, which amounts to 18 basis functions.                                       
mol = gto.M(
    atom='Ar 0 0 0',
    basis='6-31G**',
    symmetry=False,
    verbose=4
)
# Atomic orbitals are characterized by                                                           
# 1s, 2s, 2p, 3s, 3p, 4s, 4p, and 3d orbitals in                                                 
# acsending order of orbital energy:                                                             
#  mo_energy =                                                                                   
#[-118.59460559  -12.31810372   -9.5684429    -9.5684429    -9.5684429                           
#   -1.27474354   -0.58891803   -0.58891803   -0.58891803    0.62666152                          
#    0.72602205    0.72602205    0.72602205    1.21762909    1.21762909                          
#    1.21762909    1.21762909    1.21762909]                                                     

System: uname_result(system='Darwin', node='MacBook-Pro-3.local', release='24.6.0', version='Darwin Kernel Version 24.6.0: Mon Jul 14 11:30:40 PDT 2025; root:xnu-11417.140.69~1/RELEASE_ARM64_T6041', machine='arm64')  Threads 1
Python 3.12.7 (main, Oct  7 2024, 18:36:03) [Clang 15.0.0 (clang-1500.3.9.4)]
numpy 1.26.4  scipy 1.17.1  h5py 3.12.1
Date: Mon Jun 15 12:31:43 2026
PySCF version 2.11.0
PySCF path  /Users/sato/.pyenv/versions/3.12.7/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 1
[INPUT] num. electrons = 18
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Ar     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0

nuclear repulsion = 0
number

In [4]:
# Freeze 1s and 2s to prepare an effective                                                       
# (16o, 14e) active space problem.                                                               
n_core = 2        # core orbitals: 1s,2s ((2,2) electrons)                                       
n_act = 16        # active orbitals: 2p,3s,3p, and 4s,4p,3d                                      
n_virt = 0        # no virtual orbitals                                                          
n_elec = (7,7)    # active electrons, each spin                                                  
n_elec_total = 14 # active electrons

In [5]:
# Restricted Active Space (RAS) example:                                                         
# RAS1 (2p, maximam hole = 2)                                                                    
# RAS2 (3s, 3p, 4s, 4p)                                                                          
# RAS3 (3d, maximam particle = 2)                                                                
occ_info_spin = [
    [
        {'n_orb':3, 'min': 1, 'max': 3}, # 2p: N = 1, 2, 3 (up to two holes)                     
        {'n_orb':8, 'min':-1, 'max':-1}, # 3s,3p,4s,4p: free occupation                          
        {'n_orb':5, 'min': 0, 'max': 2}, # 3d: N = 0, 1, 2 (up to two particles)                 
    ],
    [
        {'n_orb':3, 'min': 1, 'max': 3}, # 2p: N = 1, 2, 3 (up to two holes)                     
        {'n_orb':8, 'min':-1, 'max':-1}, # 3s,3p,4s,4p: free occupation                          
        {'n_orb':5, 'min': 0, 'max': 2}, # 3d: N = 0, 1, 2 (up to two particles)                 
    ]
]
occ_info_total = [
        {'n_orb': 6, 'min': 4, 'max': 6}, # 2p: N = 4, 5, 6 (up to two holes)                    
        {'n_orb':16, 'min':-1, 'max':-1}, # 3s,3p,4s,4p: free occupation                         
        {'n_orb':10, 'min': 0, 'max': 2}, # 3d: N = 0, 1, 2 (up to two particles)                
]

In [6]:
########################################################################                         
# CI instance generation                                                                         
print("\n")
print("#"*72)
print("[Step 1] Constructing CI object...")
t0 = time.time()
myCI = ORMAS2(n_elec,
              occ_info_spin,
              occ_info_total,
              num_threads = 10,
              verbose = 1)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"CI dimension: {myCI.total_dim}")
print("String distribution over occupation groups:")
print(myCI.mat_num_str)



########################################################################
[Step 1] Constructing CI object...
ORMAS1: num_threads =  10
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 0.507 seconds
ORMAS1: build_string_to_index 0.341 seconds
ORMAS1: _generate_trans1 1.831 seconds
ORMAS1: _generate_trans2 1.024 seconds
ORMAS1: num_threads =  10
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 0.008 seconds
ORMAS1: build_string_to_index 0.002 seconds
ORMAS1: _generate_trans1 0.010 seconds
ORMAS1: _generate_trans2 0.413 seconds
ORMAS1 objects construction 5.004 seconds
generate_distributions 0.001 seconds
generate_occupation_boundaries 0.000 seconds
generate_determinant_offsets 0.010 seconds
generate_transpose_map 0.004 seconds
generate_trans1_det_boundaries 0.216 seconds
generate_trans2_det_boundaries 0.858 seconds
generate_trans1_for_pq_valid 0.408 seconds
 ... Done: 6.84 sec.
CI dimension: 4379424
String distribution over occupation groups:
[[  

In [7]:
########################################################################                         
# Hamiltonian matrix elements generation                                                         
print("\n")
print("#"*72)
print("[Step 2] Running RHF to obtain MO integrals...")
t0 = time.time()
e_core, h1eff, h2_phys = get_integrals_rhf(mol, n_core, n_act, n_elec_total)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"Core energy: {e_core}")



########################################################################
[Step 2] Running RHF to obtain MO integrals...
RHF Total Energy: -526.7721510920 Hartree (Time: 0.01 sec)
Integral transformation completed (Time: 0.00 sec)
 ... Done: 0.02 sec.
Core energy: -376.5513080127919


In [8]:
########################################################################                         
# Direct-CI Davidson diagonalization
print("\n")
print("#"*72)
if myCI.total_dim < max_dim_ormas:
    print("[Step 3] ORMAS Davidson diagonalization...")
    t0 = time.time()
    def my_get_sigma(x):
        return myCI.h_prod(h1eff, h2_phys, x).real
        #return myCI.h_prod_force_symmetric(h1eff, h2_phys, x).real                              
        return get_sigma(x).real

    hdiag = myCI.calc_hdiag(h1eff, h2_phys).real
    def my_precond(dx, e, x0):
        denom = hdiag - e
        denom[abs(denom) < 1e-8] = 1e-8  # avoid zero division                                   
        return dx / denom

    E1, U1 = davidson(myCI.total_dim,
                      my_get_sigma,
                      my_precond,
                      verbose=5)
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"ORMAS CI energy: {E1 + e_core}")
else:
    print("[Step 3] myCI.total_dim too large. Skip ORMAS diagonalization.")




########################################################################
[Step 3] ORMAS Davidson diagonalization...
verbose = 5
davidson 0 1  |r|= 0.733  e= [-150.22084308]  max|de|= -150  lindep=    1
davidson 1 2  |r|= 0.156  e= [-150.36621649]  max|de|= -0.145  lindep= 0.999
davidson 2 3  |r|= 0.0339  e= [-150.37087279]  max|de|= -0.00466  lindep= 0.997
davidson 3 4  |r|= 0.00717  e= [-150.37111646]  max|de|= -0.000244  lindep= 0.996
davidson 4 5  |r|= 0.00164  e= [-150.3711254]  max|de|= -8.95e-06  lindep= 0.986
davidson 5 6  |r|= 0.000434  e= [-150.37112598]  max|de|= -5.72e-07  lindep= 0.917
davidson 6 7  |r|= 0.0001  e= [-150.37112601]  max|de|= -3.6e-08  lindep= 0.947
davidson 7 8  |r|= 2.11e-05  e= [-150.37112601]  max|de|= -2e-09  lindep= 0.981
davidson 8 9  |r|= 4.33e-06  e= [-150.37112601]  max|de|= -7.59e-11  lindep= 0.921
davidson 9 10  |r|= 1.02e-06  e= [-150.37112601]  max|de|= -3.72e-12  lindep= 0.927
root 0 converged  |r|= 1.94e-07  e= -150.37112601345748  max|de|= 